In [1]:
!pip install -q hf_xet
# Do NOT `pip install -U datasets` here: it swaps pyarrow under the running kernel and
# breaks with "pyarrow.lib.Array size changed". Kaggle's preinstalled datasets is enough.
# Plan task 9: a more realistic ReRAM crossbar (single Kaggle notebook cell).
#
# Extends the task 6-8 simulator (reram_sim/kaggle/kaggle_reram_sweep.py, unchanged) with:
#   fixed ADC        full-scale range calibrated once per layer and slice on a few training
#                    sentences with a noise-free chip, then kept fixed (the old ADC re-scaled
#                    itself on every batch, which a real chip cannot do)
#   bit-serial input inputs applied one magnitude bit at a time (7 bits, sign-magnitude: a
#                    negative input drives a negative read voltage), each bit's column
#                    current read by the ADC, then shift-and-add
#   read noise       fresh Gaussian noise on every cell at every read
#   level noise      programming noise that depends on the level a cell is programmed to
#                    (e.g. the middle MLC levels are less stable)
#   clustered faults stuck cells in k x k clusters, or whole broken bit lines
#   IR drop          static first-order model: each cell sees the wire resistance between it
#                    and the driver / sense amplifier, g_eff = g / (1 + r * g * n_segments)
#
# Setup:
#   1. Settings -> Accelerator: GPU T4, Internet: On.
#   2. Add Input -> the svd-tiering notebook (holds outputs/<model>/ with svd_model.pt etc.).
#   3. Paste this whole file into ONE notebook cell. Run EXPERIMENT = "smoke" first: it runs
#      the simulator self-tests and a tiny sweep. Then pick an experiment below.
# Results: /kaggle/working/reram_results/<experiment>/<model>/{sweep.csv, sweep_summary.csv,
#          sweep.png, config.json}
#
# Cell model: 8-bit signed weights per row, differential pair (G+, G-), bit-sliced into
# base-2^b digits (SLC b=1, MLC b=2). Digit d -> g = g_min + d*delta, G_max = 1, on/off 100.
# Physical layout: crossbar rows = layer inputs, columns = layer outputs, tiles of
# tile_rows x tile_cols cells. Runs in fp32 on purpose (fp16 overflows the partial sums).
import copy
import csv
import json
import math
import shutil
import time
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from statistics import mean, pstdev

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

# ---------------------------------------------------------------- config
# Pick ONE experiment. Each sweeps one parameter ("param") over "values", with every other
# crossbar setting taken from "fixed" (and the defaults in CrossbarConfig).
#   "smoke"      self-tests + a tiny noise-free DistilBERT sweep (128 sentences); expect every
#                placement to match the digital SVD model. A few minutes.
#   "adc"        ADC resolution with bit-serial inputs, no noise (deterministic, 1 chip)
#   "adc_noise"  ADC resolution with bit-serial inputs at sigma = 5% (5 chips)
#   "read"       read noise, no programming noise
#   "level"      level-dependent MLC programming noise at sigma = 5%
#   "cluster"    1% stuck cells: scattered vs 3x3 clusters vs whole bit lines
#   "irdrop"     wire resistance (R_wire / R_on per cell segment), no noise
#   "all"        self-tests, then every experiment above except smoke, cheapest first
#                (about 6-8 GPU hours; results are zipped after each experiment)
EXPERIMENT = "all"
ALL_ORDER = ["irdrop", "read", "cluster", "level", "adc", "adc_noise"]

BASE_CONFIG = {
    "models": ["distilbert-sst2", "bert-base-sst2"],
    "policies": ["all_slc", "all_mlc", "magnitude", "gradient"],
    "slc_frac": 0.2,           # share of components in SLC for magnitude / gradient
    "seeds": 5,                # simulated chips per point (1 is used when a point has no randomness)
    "slc_bits": 1,
    "mlc_bits": 2,
    "adc_calib_n": 64,         # SST-2 train sentences used to calibrate the fixed ADC range
    "eval_n": None,            # None = all 872 validation sentences
    "batch_size": 64,
    "selftest": True,
    "out": Path("/kaggle/working/reram_results") if Path("/kaggle/working").exists() else Path("reram_results"),
}

PRESETS = {
    # no noise: every placement should match the digital SVD model at "ideal" and stay close at 8 bits
    "smoke": {"models": ["distilbert-sst2"], "eval_n": 128, "seeds": 1, "adc_calib_n": 16,
              "param": "adc_bits", "values": [None, 8],
              "fixed": {"input_mode": "bit_serial"}},
    # "ideal" (None) is computed with the fast readout: without an ADC it gives the same result
    "adc": {"param": "adc_bits", "values": [None, 8, 7, 6, 5, 4, 3], "seeds": 1,
            "fixed": {"input_mode": "bit_serial"}},
    # bit-serial reads are slow, so the noisy ADC run uses DistilBERT and 3 chips
    "adc_noise": {"models": ["distilbert-sst2"], "param": "adc_bits", "values": [None, 8, 6, 4], "seeds": 3,
                  "fixed": {"input_mode": "bit_serial", "sigma": 0.05}},
    "read": {"param": "read_sigma", "values": [0.0, 0.005, 0.01, 0.02, 0.03, 0.05], "seeds": 5,
             "fixed": {}},
    # multiplier on sigma for each MLC level 0..3 (SLC stays uniform)
    "level": {"param": "mlc_level_sigma", "values": [(1, 1, 1, 1), (1, 1.5, 1.5, 1), (1, 2, 2, 1)],
              "seeds": 10, "fixed": {"sigma": 0.05}},
    "cluster": {"param": "fault_mode", "values": ["uniform", "cluster", "line"], "seeds": 10,
                "fixed": {"fault_rate": 0.01}},
    "irdrop": {"param": "ir_ratio", "values": [0.0, 1e-4, 3e-4, 1e-3, 3e-3], "seeds": 1,
               "fixed": {}},
}
CONFIG = {}


def set_experiment(name):
    CONFIG.clear()
    CONFIG.update(BASE_CONFIG)
    CONFIG.update(PRESETS[name])

MODELS = {
    "distilbert-sst2": "distilbert-base-uncased-finetuned-sst-2-english",
    "bert-base-sst2": "textattack/bert-base-uncased-SST-2",
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T_START = time.time()


def log(msg):
    print(f"[{time.time() - T_START:7.1f}s] {msg}", flush=True)


# ---------------------------------------------------------------- crossbar
@dataclass(frozen=True)
class CrossbarConfig:
    bits_per_cell: int = 1
    sigma: float = 0.0                 # programming noise / G_max
    level_sigma: tuple | None = None   # per-level multiplier on sigma, len = 2**bits_per_cell
    fault_rate: float = 0.0            # expected share of stuck cells
    sa1_frac: float = 0.5              # share of stuck cells stuck at G_max (rest at G_min)
    fault_mode: str = "uniform"        # "uniform" | "cluster" (k x k blocks) | "line" (whole bit line)
    cluster: int = 3
    read_sigma: float = 0.0            # read noise / G_max, fresh at every read
    ir_ratio: float = 0.0              # R_wire per cell segment / R_on
    on_off: float = 100.0
    w_bits: int = 8
    in_bits: int = 8
    input_mode: str = "dac"            # "dac" (whole input at once) | "bit_serial"
    adc_bits: int | None = None        # None = ideal readout
    tile_rows: int = 128
    tile_cols: int = 128

    @property
    def levels(self):
        return 2 ** self.bits_per_cell

    @property
    def n_slices(self):
        return math.ceil((self.w_bits - 1) / self.bits_per_cell)


def quantize_symmetric(x, bits, dim=None):
    qmax = 2 ** (bits - 1) - 1
    absmax = x.abs().amax() if dim is None else x.abs().amax(dim=dim, keepdim=True)
    scale = absmax.clamp(min=1e-12) / qmax
    return torch.round(x / scale).clamp(-qmax, qmax), scale


def fault_masks(shape, cfg, gen, device):
    """Boolean masks (stuck at G_max, stuck at G_min) for one [out, in] cell array."""
    rand = lambda: torch.rand(shape, generator=gen, device=device)
    if cfg.fault_mode == "uniform":
        stuck, sa1 = rand() < cfg.fault_rate, rand() < cfg.sa1_frac
        return stuck & sa1, stuck & ~sa1
    if cfg.fault_mode == "cluster":
        k = cfg.cluster
        seed_rate = cfg.fault_rate / (k * k)

        def grow(seeds):
            return F.max_pool2d(seeds.float()[None, None], k, stride=1, padding=k // 2)[0, 0].bool()

        m1 = grow(rand() < seed_rate * cfg.sa1_frac)
        m0 = grow(rand() < seed_rate * (1 - cfg.sa1_frac)) & ~m1
        return m1, m0
    if cfg.fault_mode == "line":
        # one bit line = one output column inside one row tile; a broken line fails as a whole
        out, n_in = shape
        n_tiles = math.ceil(n_in / cfg.tile_rows)
        line = torch.rand((out, n_tiles), generator=gen, device=device) < cfg.fault_rate
        sa1 = torch.rand((out, n_tiles), generator=gen, device=device) < cfg.sa1_frac
        expand = lambda m: m.repeat_interleave(cfg.tile_rows, dim=1)[:, :n_in]
        return expand(line & sa1), expand(line & ~sa1)
    raise ValueError(cfg.fault_mode)


class Crossbar(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.detach().clone().float())
        for name in ("scale", "w_hat", "digits"):
            self.register_buffer(name, None, persistent=False)
        self.cfg = None
        self.fs = None             # fixed ADC full scale per slice (set by calibration)
        self.calibrating = False
        self.fs_obs = None

    def segments(self, device):
        """Wire segments between each cell and its driver (row start) and sense amp (column end)."""
        o = torch.arange(self.out_features, device=device) % self.cfg.tile_cols
        i = torch.arange(self.in_features, device=device) % self.cfg.tile_rows
        return (o[:, None] + 1) + (self.cfg.tile_rows - 1 - i)[None, :]

    @torch.no_grad()
    def program(self, cfg, gen=None):
        self.cfg = cfg
        dev = self.weight.device
        q, self.scale = quantize_symmetric(self.weight, cfg.w_bits, dim=1)
        mag = {"pos": q.clamp(min=0), "neg": (-q).clamp(min=0)}
        L = cfg.levels
        g_min = 1.0 / cfg.on_off
        delta = (1.0 - g_min) / (L - 1)
        nseg = self.segments(dev) if cfg.ir_ratio > 0 else None
        lvl = None if cfg.level_sigma is None else torch.tensor(cfg.level_sigma, device=dev, dtype=torch.float32)
        digits = []
        for s in range(cfg.n_slices):
            g = {}
            for k, m in mag.items():
                d = torch.div(m, L ** s, rounding_mode="floor") % L
                c = g_min + d * delta
                if cfg.sigma > 0:
                    sig = cfg.sigma if lvl is None else cfg.sigma * lvl[d.long()]
                    c = (c + sig * torch.randn(c.shape, generator=gen, device=dev)).clamp(min=0)
                if cfg.fault_rate > 0:
                    m1, m0 = fault_masks(c.shape, cfg, gen, dev)
                    c = torch.where(m1, torch.ones_like(c), torch.where(m0, torch.full_like(c, g_min), c))
                if nseg is not None:
                    c = c / (1 + cfg.ir_ratio * c * nseg)
                g[k] = c
            digits.append((g["pos"] - g["neg"]) / delta)
        self.delta = delta
        self.w_hat = sum(L ** s * d for s, d in enumerate(digits)) * self.scale
        self.lsum2 = sum(L ** (2 * s) for s in range(cfg.n_slices))
        self.digits = torch.stack(digits) if self.needs_slow() or self.calibrating else None

    def needs_slow(self):
        return self.cfg.adc_bits is not None or self.cfg.input_mode == "bit_serial"

    def read_noise_k(self):
        return math.sqrt(2) * self.cfg.read_sigma / self.delta     # two cells per weight

    def _fast(self, xi):
        y = F.linear(xi, self.w_hat)
        if self.cfg.read_sigma > 0:
            std = self.read_noise_k() * math.sqrt(self.lsum2) * xi.norm(dim=-1, keepdim=True)
            y = y + torch.randn_like(y) * std * self.scale.squeeze(1)
        return y

    def _planes(self, xt):
        if self.cfg.input_mode != "bit_serial":
            return [(1.0, xt)]
        # sign-magnitude: bit k of |x| applied with the sign of x (+1 / 0 / -1 read voltage).
        # Two's complement would switch on every bit of a small negative input, so read noise
        # and ADC range would be set by the sign bit rather than by the input's size.
        mag, sign = xt.abs().long(), torch.sign(xt)
        return [(2.0 ** k, ((mag >> k) & 1).float() * sign) for k in range(self.cfg.in_bits - 1)]

    def _slow(self, xi):
        cfg = self.cfg
        R = cfg.tile_rows
        n_tiles = math.ceil(self.in_features / R)
        pad = n_tiles * R - self.in_features
        xt = F.pad(xi, (0, pad)).unflatten(-1, (n_tiles, R))
        planes = self._planes(xt)
        qmax = None if cfg.adc_bits is None else 2 ** (cfg.adc_bits - 1) - 1
        y = 0
        for s, d in enumerate(self.digits):
            dt = F.pad(d, (0, pad)).unflatten(-1, (n_tiles, R))
            for wk, p in planes:
                part = torch.einsum("...tr,otr->...ot", p, dt)
                if cfg.read_sigma > 0:
                    std = self.read_noise_k() * (p * p).sum(-1).sqrt().unsqueeze(-2)
                    part = part + torch.randn_like(part) * std
                if self.calibrating:
                    self.fs_obs[s] = max(self.fs_obs[s], part.abs().amax().item())
                elif qmax is not None:
                    step = max(self.fs[s], 1e-12) / qmax
                    part = torch.round(part / step).clamp(-qmax, qmax) * step
                y = y + wk * cfg.levels ** s * part.sum(-1)
        return y * self.scale.squeeze(1)

    def forward(self, x):
        xi, x_scale = quantize_symmetric(x, self.cfg.in_bits)
        y = self._slow(xi) if (self.needs_slow() or self.calibrating) else self._fast(xi)
        return y * x_scale


class ReRAMLinear(nn.Module):
    """y = U_slc (SVh_slc x) + U_mlc (SVh_mlc x) + bias, each factor its own crossbar."""

    def __init__(self, U, SVh, slc_mask, bias):
        super().__init__()
        self.tiers = nn.ModuleDict()
        for tier, mask in (("SLC", slc_mask), ("MLC", ~slc_mask)):
            idx = mask.nonzero().flatten()
            if idx.numel():
                self.tiers[tier] = nn.Sequential(Crossbar(SVh[idx]), Crossbar(U[:, idx]))
        self.register_buffer("bias", None if bias is None else bias.detach().clone().float())

    def program(self, cfgs, gen):
        for tier, pair in self.tiers.items():
            for xbar in pair:
                xbar.program(cfgs[tier], gen)

    def forward(self, x):
        y = sum(pair(x) for pair in self.tiers.values())
        return y if self.bias is None else y + self.bias


def crossbars(model):
    return [m for m in model.modules() if isinstance(m, Crossbar)]


def program_model(model, cfgs, seed):
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    torch.manual_seed(seed)                       # read noise uses the global generator
    for m in model.modules():
        if isinstance(m, ReRAMLinear):
            m.program(cfgs, gen)


# ---------------------------------------------------------------- self-tests
def selftest():
    """Checks the new simulator pieces on random matrices. Raises if any check fails."""
    log("[self-test] simulator checks on random 300x260 matrices")
    torch.manual_seed(0)
    gen = lambda: torch.Generator(device=DEVICE).manual_seed(1)
    W = torch.randn(300, 260, device=DEVICE)
    x = torch.randn(64, 260, device=DEVICE)
    results = []

    def check(name, ok, detail):
        results.append(ok)
        log(f"  {'PASS' if ok else 'FAIL'}  {name}: {detail}")

    def rel(a, b):
        return ((a - b).norm() / b.norm()).item()

    q, sc = quantize_symmetric(W, 8, dim=1)
    xi, xs = quantize_symmetric(x, 8)
    ref = F.linear(xi, q * sc) * xs
    for bits in (1, 2):
        cb = Crossbar(W).to(DEVICE)
        cb.program(CrossbarConfig(bits_per_cell=bits), gen())
        # g_min offset cancels in the differential pair, so the noise-free crossbar is exact
        check(f"ideal crossbar = INT8 matmul ({bits} b/cell)", rel(cb(x), ref) < 1e-4, f"rel err {rel(cb(x), ref):.1e}")
        for mode in ("dac", "bit_serial"):
            cfg = CrossbarConfig(bits_per_cell=bits, input_mode=mode)
            cb.calibrating, cb.fs_obs = True, [0.0] * cfg.n_slices     # keeps per-slice digits, no ADC
            cb.program(cfg, gen())
            y = cb._slow(xi) * xs
            cb.calibrating = False
            check(f"{mode} slice-by-slice readout = ideal ({bits} b/cell)", rel(y, ref) < 1e-4, f"rel err {rel(y, ref):.1e}")

    # bit-serial decomposition is exact
    xi = quantize_symmetric(x, 8)[0]
    cb = Crossbar(W).to(DEVICE)
    cb.program(CrossbarConfig(input_mode="bit_serial"), gen())
    rebuilt = sum(wk * p for wk, p in cb._planes(xi))
    check("sign-magnitude bit planes rebuild the input", torch.equal(rebuilt, xi), "exact")

    # fixed ADC: calibrate, then a 16-bit ADC is near-lossless and a 4-bit ADC is not
    for mode in ("dac", "bit_serial"):
        cfg = CrossbarConfig(bits_per_cell=2, input_mode=mode, adc_bits=16)
        cb = Crossbar(W).to(DEVICE)
        cb.calibrating, cb.fs_obs = True, [0.0] * cfg.n_slices
        cb.program(cfg, gen())
        cb(x)
        cb.calibrating, cb.fs = False, list(cb.fs_obs)
        e16 = rel(cb(x), ref)
        cb.program(replace(cfg, adc_bits=4), gen())
        e4 = rel(cb(x), ref)
        check(f"fixed ADC ({mode}): 16 bit ~ lossless, 4 bit lossy", e16 < 1e-3 < e4, f"16b {e16:.1e}, 4b {e4:.1e}")

    # read noise: measured spread matches the formula, in both readout paths
    for mode in ("dac", "bit_serial"):
        cfg = CrossbarConfig(bits_per_cell=2, input_mode=mode, read_sigma=0.02)
        cb = Crossbar(W).to(DEVICE)
        cb.program(cfg, gen())
        meas = (torch.stack([cb(x) for _ in range(20)]) - ref).std().item()
        if mode == "dac":
            pred_i = cb.read_noise_k() * math.sqrt(cb.lsum2) * xi.norm(dim=-1, keepdim=True)
        else:
            planes = cb._planes(xi)
            pred_i = cb.read_noise_k() * torch.sqrt(sum(wk ** 2 * (p * p).sum(-1, keepdim=True) for wk, p in planes)
                                                    * cb.lsum2)
        pred = (pred_i * cb.scale.squeeze(1) * xs).pow(2).mean().sqrt().item()
        check(f"read noise spread matches formula ({mode})", abs(meas / pred - 1) < 0.15,
              f"measured {meas:.4f}, predicted {pred:.4f}")

    # fault modes give the requested share of stuck cells
    for mode in ("uniform", "cluster", "line"):
        cfg = CrossbarConfig(fault_rate=0.02, fault_mode=mode)
        m1, m0 = fault_masks((1024, 1024), cfg, gen(), DEVICE)
        frac = (m1 | m0).float().mean().item()
        check(f"{mode} faults hit ~2% of cells", 0.014 < frac < 0.026, f"{100 * frac:.2f}%")

    # IR drop: zero resistance changes nothing; positive resistance shrinks the weights
    cb = Crossbar(W).to(DEVICE)
    cb.program(CrossbarConfig(ir_ratio=0.0), gen())
    w0 = cb.w_hat.clone()
    cb.program(CrossbarConfig(ir_ratio=1e-3), gen())
    shrink = (cb.w_hat.abs().sum() / w0.abs().sum()).item()
    check("IR drop shrinks stored weights", 0.5 < shrink < 1.0, f"|W| ratio {shrink:.3f}")

    # level-dependent noise: uniform multipliers equal plain sigma
    cb = Crossbar(W).to(DEVICE)
    cb.program(CrossbarConfig(bits_per_cell=2, sigma=0.05), gen())
    a = cb.w_hat.clone()
    cb.program(CrossbarConfig(bits_per_cell=2, sigma=0.05, level_sigma=(1, 1, 1, 1)), gen())
    check("level noise with all-1 multipliers = plain noise", torch.allclose(a, cb.w_hat), "same chip")

    if not all(results):
        raise RuntimeError("simulator self-test failed; see FAIL lines above")
    log(f"[self-test] all {len(results)} checks passed")


# ---------------------------------------------------------------- model loading
class SVDLinear(nn.Module):
    """Shell matching svd_tiering's SVDLinear so svd_model.pt loads into it."""

    def __init__(self, m, k, n, has_bias):
        super().__init__()
        self.rank = k
        self.U = nn.Parameter(torch.zeros(m, k))
        self.S = nn.Parameter(torch.zeros(k))
        self.Vh = nn.Parameter(torch.zeros(k, n))
        self.bias = nn.Parameter(torch.zeros(m)) if has_bias else None

    def forward(self, x):
        return F.linear(F.linear(x, self.Vh) * self.S, self.U, self.bias)


def replace_modules(model, build):
    for name, new in build.items():
        parent, _, child = name.rpartition(".")
        setattr(model.get_submodule(parent) if parent else model, child, new)


def find_svd_outputs(key):
    for p in Path("/kaggle/input").rglob("tier_map.json") if Path("/kaggle/input").exists() else []:
        if p.parent.name == key:
            return p.parent
    for z in Path("/kaggle/input").rglob("outputs.zip") if Path("/kaggle/input").exists() else []:
        dest = Path("/kaggle/working/svd_outputs")
        if not (dest / key / "tier_map.json").exists():
            log(f"  unzipping {z} -> {dest}")
            zipfile.ZipFile(z).extractall(dest)
        if (dest / key / "tier_map.json").exists():
            return dest / key
    raise FileNotFoundError(f"no {key}/tier_map.json under /kaggle/input; attach the svd-tiering notebook")


def load_svd_model(key, src):
    tok = AutoTokenizer.from_pretrained(MODELS[key])
    model = AutoModelForSequenceClassification.from_pretrained(MODELS[key])
    tier_map = json.loads((src / "tier_map.json").read_text())
    shells = {name: SVDLinear(e["out_features"], e["rank"], e["in_features"],
                              model.get_submodule(name).bias is not None)
              for name, e in tier_map.items()}
    replace_modules(model, shells)
    model.load_state_dict(torch.load(src / "svd_model.pt", map_location="cpu"))
    return model.eval(), tok, shells


def tier_masks(layers, importance, policy, frac):
    if policy == "all_slc":
        return {n: torch.ones(l.rank, dtype=torch.bool) for n, l in layers.items()}
    if policy == "all_mlc":
        return {n: torch.zeros(l.rank, dtype=torch.bool) for n, l in layers.items()}
    masks = {}
    for n, s in importance[policy].items():
        mask = torch.zeros_like(s, dtype=torch.bool)
        mask[torch.topk(s, math.ceil(frac * s.numel())).indices] = True
        masks[n] = mask
    return masks


def to_reram(model, layers, masks):
    built = {}
    for name, svd in layers.items():
        U, SVh = svd.U.detach(), (svd.S[:, None] * svd.Vh).detach()
        built[name] = ReRAMLinear(U, SVh, masks[name], svd.bias)
    replace_modules(model, built)


def sst2_batches(tok, split, n, seed):
    ds = load_dataset("nyu-mll/glue", "sst2", split=split)
    if n is not None and n < len(ds):
        ds = ds.shuffle(seed=seed).select(range(n))
    ds = ds.map(lambda b: tok(b["sentence"], truncation=True, max_length=128),
                batched=True, remove_columns=["sentence", "idx"]).rename_column("label", "labels")
    loader = DataLoader(ds, batch_size=CONFIG["batch_size"], collate_fn=DataCollatorWithPadding(tok))
    return [{k: v.to(DEVICE) for k, v in b.items()} for b in loader]


@torch.no_grad()
def evaluate(model, batches):
    correct = total = 0
    for b in batches:
        logits = model(**b).logits
        correct += (logits.argmax(-1) == b["labels"]).sum().item()
        total += b["labels"].numel()
    return correct / total


@torch.no_grad()
def calibrate_adc(model, cfgs, calib):
    """Fix each crossbar's ADC full scale per slice: the largest partial sum seen on a
    noise-free chip over the calibration sentences."""
    xbars = crossbars(model)
    for cb in xbars:
        cb.calibrating = True
    clean = {t: replace(c, sigma=0.0, fault_rate=0.0, read_sigma=0.0, adc_bits=None) for t, c in cfgs.items()}
    program_model(model, clean, 0)
    for cb in xbars:
        cb.fs_obs = [0.0] * cb.cfg.n_slices
    for b in calib:
        model(**b)
    for cb in xbars:
        cb.calibrating, cb.fs = False, list(cb.fs_obs)
        cb.digits = None
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ---------------------------------------------------------------- sweep
def tier_cfgs(value):
    """SLC / MLC crossbar configs for one sweep value."""
    fixed = dict(CONFIG["fixed"])
    cfgs = {"SLC": CrossbarConfig(bits_per_cell=CONFIG["slc_bits"], **fixed),
            "MLC": CrossbarConfig(bits_per_cell=CONFIG["mlc_bits"], **fixed)}
    if CONFIG["param"] == "mlc_level_sigma":
        cfgs["MLC"] = replace(cfgs["MLC"], level_sigma=tuple(value))
    else:
        cfgs = {t: replace(c, **{CONFIG["param"]: value}) for t, c in cfgs.items()}
    if CONFIG["param"] == "adc_bits" and value is None:
        cfgs = {t: replace(c, input_mode="dac") for t, c in cfgs.items()}   # same result, much faster
    return cfgs


def is_random(cfgs):
    return any(c.sigma > 0 or c.fault_rate > 0 or c.read_sigma > 0 for c in cfgs.values())


def label(v):
    if v is None:
        return "ideal"
    if isinstance(v, (tuple, list)):
        return "x" + "/".join(f"{a:g}" for a in v)
    return f"{v:g}" if isinstance(v, float) else str(v)


def jobs():
    out = []
    for p in CONFIG["policies"]:
        out.append((p, 1.0 if p == "all_slc" else 0.0 if p == "all_mlc" else CONFIG["slc_frac"]))
    return out


def plot(summary, digital, path, key):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    labels = [label(v) for v in CONFIG["values"]]
    fig, ax = plt.subplots(figsize=(7, 4))
    for policy, frac in jobs():
        rows = {r["value"]: r for r in summary if r["policy"] == policy}
        ys = [rows[l]["acc_mean"] if l in rows else float("nan") for l in labels]
        es = [rows[l]["acc_std"] if l in rows else 0 for l in labels]
        name = policy if policy.startswith("all") else f"{policy} {100 * frac:.0f}% SLC"
        ax.errorbar(range(len(labels)), ys, yerr=es, marker="o", capsize=3, label=name)
    ax.axhline(digital, color="gray", ls="--", lw=1, label="digital SVD model")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_xlabel(CONFIG["param"])
    ax.set_ylabel("SST-2 accuracy")
    fixed = ", ".join(f"{k}={v}" for k, v in CONFIG["fixed"].items()) or "no other non-idealities"
    ax.set_title(f"{key}: {EXPERIMENT} ({fixed})", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()
    log(f"  plot -> {path}")


def run_model(key):
    out = CONFIG["out"] / EXPERIMENT / key
    out.mkdir(parents=True, exist_ok=True)
    log(f"========== {key} (experiment: {EXPERIMENT}, sweep {CONFIG['param']}) ==========")

    log("[1/6] locating SVD-tiering outputs")
    src = find_svd_outputs(key)
    log(f"  {src}")

    log("[2/6] loading SVD model, importance scores and SST-2")
    digital_model, tok, layers = load_svd_model(key, src)
    importance = {m: {n: s.cpu().float() for n, s in d.items()}
                  for m, d in torch.load(src / "importance.pt", map_location="cpu").items()}
    batches = sst2_batches(tok, "validation", CONFIG["eval_n"], 0)
    log(f"  {len(layers)} factorised layers, {sum(len(b['labels']) for b in batches)} validation sentences")
    needs_adc = any(tier_cfgs(v)["SLC"].adc_bits is not None for v in CONFIG["values"])
    calib = sst2_batches(tok, "train", CONFIG["adc_calib_n"], 2) if needs_adc else None
    if needs_adc:
        log(f"  {CONFIG['adc_calib_n']} training sentences for ADC calibration")

    log("[3/6] digital reference (SVD model, FP32)")
    digital = evaluate(digital_model.to(DEVICE), batches)
    digital_model.cpu()
    log(f"  digital SVD model acc {digital:.4f}")

    with open(out / "config.json", "w") as f:
        json.dump({"experiment": EXPERIMENT, "config": {k: str(v) for k, v in CONFIG.items()},
                   "source": str(src), "digital_acc": digital,
                   "crossbar": {label(v): {t: asdict(c) for t, c in tier_cfgs(v).items()} for v in CONFIG["values"]}},
                  f, indent=2, default=str)

    n_seeds = {label(v): CONFIG["seeds"] if is_random(tier_cfgs(v)) else 1 for v in CONFIG["values"]}
    n_points = len(jobs()) * sum(n_seeds.values())
    log(f"[4/6] sweep: {len(jobs())} placements x {len(CONFIG['values'])} values of {CONFIG['param']} "
        f"= {n_points} evaluations (1 chip where nothing is random)")
    rows, done, t0 = [], 0, time.time()
    with open(out / "sweep.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["policy", "slc_frac", "param", "value", "seed", "acc"])
        for policy, frac in jobs():
            model = copy.deepcopy(digital_model)
            masks = tier_masks(layers, importance, policy, frac)
            to_reram(model, layers, masks)
            model.to(DEVICE).eval()
            n_slc = sum(int(m.sum()) for m in masks.values())
            log(f"  placement {policy} ({n_slc}/{sum(m.numel() for m in masks.values())} components in SLC)")
            if needs_adc:
                t = time.time()
                calibrate_adc(model, tier_cfgs(next(v for v in CONFIG["values"] if v is not None)), calib)
                log(f"    ADC calibrated in {time.time() - t:.0f}s")
            for v in CONFIG["values"]:
                cfgs = tier_cfgs(v)
                accs = []
                for seed in range(n_seeds[label(v)]):
                    program_model(model, cfgs, seed)
                    acc = evaluate(model, batches)
                    accs.append(acc)
                    rows.append({"policy": policy, "slc_frac": frac, "value": label(v), "seed": seed, "acc": acc})
                    w.writerow([policy, frac, CONFIG["param"], label(v), seed, f"{acc:.6f}"])
                    done += 1
                f.flush()
                eta = (time.time() - t0) / done * (n_points - done)
                log(f"    [{done:4d}/{n_points}] {policy:9s} {CONFIG['param']}={label(v):12s} "
                    f"acc {mean(accs):.4f} +- {pstdev(accs):.4f} (min {min(accs):.4f})  eta {eta / 60:.0f} min")
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    log("[5/6] summary")
    groups = defaultdict(list)
    for r in rows:
        groups[(r["policy"], r["slc_frac"], r["value"])].append(r["acc"])
    summary = [{"policy": p, "slc_frac": fr, "param": CONFIG["param"], "value": v, "acc_mean": mean(a),
                "acc_std": pstdev(a), "acc_min": min(a), "n": len(a)} for (p, fr, v), a in groups.items()]
    with open(out / "sweep_summary.csv", "w", newline="") as f:
        dw = csv.DictWriter(f, fieldnames=list(summary[0]))
        dw.writeheader()
        dw.writerows(summary)
    labels = [label(v) for v in CONFIG["values"]]
    at = {(r["policy"], r["value"]): r["acc_mean"] for r in summary}
    log(f"  digital SVD model {digital:.4f}")
    log(f"  {CONFIG['param']:>18s} " + " ".join(f"{l:>12s}" for l in labels))
    for policy, _ in jobs():
        log(f"  {policy:>18s} " + " ".join(f"{100 * at[(policy, l)]:12.2f}" for l in labels))

    log("[6/6] plot")
    plot(summary, digital, out / "sweep.png", key)


log(f"device: {DEVICE} {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else ''}")
log(f"torch {torch.__version__}")
RUN = ALL_ORDER if EXPERIMENT == "all" else [EXPERIMENT]
log(f"experiments to run: {RUN}")
if BASE_CONFIG["selftest"]:
    selftest()                     # stops here if the simulator is wrong, before using GPU hours
failed = []
for EXPERIMENT in RUN:
    set_experiment(EXPERIMENT)
    log(f"################ experiment {EXPERIMENT} ({RUN.index(EXPERIMENT) + 1}/{len(RUN)}) ################")
    log(f"config: {CONFIG}")
    try:
        for key in CONFIG["models"]:
            run_model(key)
    except Exception as e:         # keep going with the next experiment
        failed.append(EXPERIMENT)
        log(f"!! experiment {EXPERIMENT} failed: {type(e).__name__}: {e}")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    log("zipping results so far")
    shutil.make_archive(str(BASE_CONFIG["out"]), "zip", BASE_CONFIG["out"])

log(f"finished. failed experiments: {failed or 'none'}")
log(f"download {BASE_CONFIG['out']}.zip from the Output panel")


[    0.1s] device: cuda Tesla T4
[    0.1s] torch 2.11.0+cu128
[    0.1s] experiments to run: ['irdrop', 'read', 'cluster', 'level', 'adc', 'adc_noise']
[    0.1s] [self-test] simulator checks on random 300x260 matrices
[    0.8s]   PASS  ideal crossbar = INT8 matmul (1 b/cell): rel err 1.7e-07
[    0.9s]   PASS  dac slice-by-slice readout = ideal (1 b/cell): rel err 1.8e-07
[    1.0s]   PASS  bit_serial slice-by-slice readout = ideal (1 b/cell): rel err 1.7e-07
[    1.0s]   PASS  ideal crossbar = INT8 matmul (2 b/cell): rel err 0.0e+00
[    1.0s]   PASS  dac slice-by-slice readout = ideal (2 b/cell): rel err 1.5e-07
[    1.0s]   PASS  bit_serial slice-by-slice readout = ideal (2 b/cell): rel err 1.5e-07
[    1.1s]   PASS  sign-magnitude bit planes rebuild the input: exact
[    1.1s]   PASS  fixed ADC (dac): 16 bit ~ lossless, 4 bit lossy: 16b 4.3e-05, 4b 2.0e-01
[    1.1s]   PASS  fixed ADC (bit_serial): 16 bit ~ lossless, 4 bit lossy: 16b 6.2e-05, 4b 2.9e-01
[    1.2s]   PASS  read n

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (42 outputs left out)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

[ 5967.2s]   64 training sentences for ADC calibration
[ 5967.2s] [3/6] digital reference (SVD model, FP32)
[ 5970.2s]   digital SVD model acc 0.9014
[ 5970.2s] [4/6] sweep: 4 placements x 7 values of adc_bits = 28 evaluations (1 chip where nothing is random)
[ 5970.7s]   placement all_slc (33168/33168 components in SLC)
[ 5983.8s]     ADC calibrated in 13s
[ 5987.5s]     [   1/28] all_slc   adc_bits=ideal        acc 0.8968 +- 0.0000 (min 0.8968)  eta 8 min
[ 6330.9s]     [   2/28] all_slc   adc_bits=8            acc 0.8991 +- 0.0000 (min 0.8991)  eta 78 min
[ 6674.3s]     [   3/28] all_slc   adc_bits=7            acc 0.9002 +- 0.0000 (min 0.9002)  eta 98 min
[ 7017.7s]     [   4/28] all_slc   adc_bits=6            acc 0.8933 +- 0.0000 (min 0.8933)  eta 105 min
[ 7361.4s]     [   5/28] all_slc   adc_bits=5            acc 0.5115 +- 0.0000 (min 0.5115)  eta 107 min
[ 7705.9s]     [   6/28] all_slc   adc_bits=4            acc 0.6112 +- 0.0000 (min 0.6112)  eta 106 min
[ 8049.8s]     [   7

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[12583.4s]   36 factorised layers, 872 validation sentences
[12584.1s]   64 training sentences for ADC calibration
[12584.1s] [3/6] digital reference (SVD model, FP32)
[12585.6s]   digital SVD model acc 0.8911
[12585.6s] [4/6] sweep: 4 placements x 4 values of adc_bits = 48 evaluations (1 chip where nothing is random)
[12585.9s]   placement all_slc (16584/16584 components in SLC)
[12592.4s]     ADC calibrated in 7s
[12598.1s]     [   3/48] all_slc   adc_bits=ideal        acc 0.8815 +- 0.0057 (min 0.8739)  eta 3 min
[13116.9s]     [   6/48] all_slc   adc_bits=8            acc 0.8830 +- 0.0034 (min 0.8784)  eta 62 min
[13635.1s]     [   9/48] all_slc   adc_bits=6            acc 0.8869 +- 0.0104 (min 0.8750)  eta 76 min
[14154.4s]     [  12/48] all_slc   adc_bits=4            acc 0.5260 +- 0.0044 (min 0.5218)  eta 78 min
[14154.8s]   placement all_mlc (0/16584 components in SLC)
[14158.6s]     ADC calibrated in 4s
[14164.0s]     [  15/48] all_mlc   adc_bits=ideal        acc 0.7783 +- 0.05